# Define Rank-Genes Zone Markers for Tangram — MASH Hepatocytes

Using all RNA-multiome donors for the **mash** condition: load the
condition's reference scRNA object, subset to hepatocytes, drop
the Stress hepatocyte subtype only (Senescent hepatocytes are kept -- they're the population of interest in MASH), downsample to `sc_cells` cells, and run
`rank_genes_groups` (by `cellsubtype`) to get zone/subtype marker genes.
Saves the full ranked gene list per subtype (one `.txt` per cell type) and
writes out a Tangram-ready reference `.h5ad` + top-`gene_nums` marker list.

**Cleanup notes (this pass):** consolidated imports (previously scattered
across several cells through the notebook) to the top; nothing else changed (this notebook was already free of dead/duplicate cells).

In [2]:
import os
import csv

import numpy as np
import pandas as pd
import scanpy as sc


## Load mash hepatocytes and drop the Stress hepatocyte subtype only (Senescent hepatocytes are kept -- they're the population of interest in MASH)

In [4]:
adata_mash = sc.read_h5ad('/tscc/projects/ps-epigen/users/cmiciano/Liver/RNA/spatial/spatial_scr/condition_h5ad/MASH.h5ad')
sc.pp.normalize_total(adata_mash)  # raw counts on load
sc.pp.log1p(adata_mash)
adata_mash_heps = adata_mash[adata_mash.obs['celltype'].isin(['Hepatocytes'])]
adata_mash_heps.obs['donor_demux'].value_counts()


donor_demux
HL221215    5625
HL221006    5105
HL170064    4009
HL170066    3715
HL230212    3495
HL200525    2127
HL220127    1946
HL170055    1943
HL191216    1675
HL200529    1661
HL200925    1505
HL170046    1419
HL210726    1327
HL170058    1281
HL191002    1139
HL160022    1097
HL191025     798
HL201024     780
HL190827     696
HL150013     541
HL221019     391
HL230302     311
HL180071     292
Name: count, dtype: int64

In [11]:
adata_mash_heps.obs['cellsubtype'].value_counts()

cellsubtype
Hepatocytes.Zone2        14148
Hepatocytes.Zone3        14090
Hepatocytes.Zone1         8039
Hepatocytes.Senescent     6083
Hepatocytes.Stress         518
Name: count, dtype: int64

In [12]:
# remove the Stress hepatocyte subtype only (Senescent hepatocytes are kept -- they're the population of interest in MASH)
adata_mash_heps = adata_mash_heps[~adata_mash_heps.obs['cellsubtype'].isin(['Hepatocytes.Stress'])]
adata_mash_heps.obs['cellsubtype'].value_counts()


cellsubtype
Hepatocytes.Zone2        14148
Hepatocytes.Zone3        14090
Hepatocytes.Zone1         8039
Hepatocytes.Senescent     6083
Name: count, dtype: int64

In [13]:
adata_mash_heps

View of AnnData object with n_obs × n_vars = 42360 × 36601
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'percent.mt', 'nCount_RNA_raw', 'nFeature_RNA_raw', 'donor_demux', 'nCount_SCT', 'nFeature_SCT', 'SCT.weight', 'seurat_clusters', 'log_nCount_SCT', 'log_nFeature_SCT', 'lane', 'batch', 'gex_raw_reads', 'gex_mapped_reads', 'gex_conf_intergenic_reads', 'gex_conf_exonic_reads', 'gex_conf_intronic_reads', 'gex_conf_exonic_unique_reads', 'gex_conf_exonic_antisense_reads', 'gex_conf_exonic_dup_reads', 'gex_exonic_umis', 'gex_conf_intronic_unique_reads', 'gex_conf_intronic_antisense_reads', 'gex_conf_intronic_dup_reads', 'gex_intronic_umis', 'gex_conf_txomic_unique_reads', 'gex_umis_count', 'gex_genes_count', 'atac_raw_reads', 'atac_unmapped_reads', 'atac_lowmapq', 'atac_dup_reads', 'atac_chimeric_reads', 'atac_mitochondrial_reads', 'atac_fragments', 'atac_TSS_fragments', 'atac_peak_region_fragments', 'atac_peak_region_cutsites', 'TSS.enrichment', 'TSS.percentile', 'condition', 'dis

## Downsample and rank genes

In [14]:
sc_cells = 10000

In [15]:
# User input: set n_obs to the number of cells to keep.
sc.pp.subsample(adata_mash_heps, n_obs=sc_cells, random_state=42)
adata_mash_heps


In [17]:
sc.tl.rank_genes_groups(adata_mash_heps, groupby="cellsubtype", use_raw=False)


## Save the full ranked gene list per cell type

In [19]:
markers_df_adata_mash_heps = pd.DataFrame(adata_mash_heps.uns["rank_genes_groups"]["names"]).iloc[:, :]


In [20]:
markers_df_adata_mash_heps

,Hepatocytes.Senescent,Hepatocytes.Zone1,Hepatocytes.Zone2,Hepatocytes.Zone3
0,DTNA,CPS1,FGG,SLCO1B3
1,SERPINE1,HAL,FGB,CYP3A4
2,FAT1,PPARGC1A,HP,SLCO1B7
3,KLHL29,SDS,SAA2,ZNF385D
4,ANXA2,INSIG1,C3,CYP2E1
...,...,...,...,...
36596,SLC38A4,ZNF385D-AS2,KLF6,FGF14
36597,PLG,LINC01344,NRG1,FGFR2
36598,ADRA1A,MPPED1,AKR1C1,CREB5
36599,ADGRA3,CYP3A4,CYP3A4,HAL


In [21]:
# Iterate through each column (cell type) and save its full ranked gene list to a text file.
for celltype in markers_df_adata_mash_heps.columns:
    genes = markers_df_adata_mash_heps[celltype].dropna()  # drop any NaN values if present
    genes_with_quotes = [f'{gene}' for gene in genes]
    output_df = pd.DataFrame(genes_with_quotes, columns=["x"])

    print(output_df.head())
    output_df.to_csv(
        f"/tscc/projects/ps-epigen/users/cmiciano/Liver/RNA/scripts/zone_gene_lists/MASH_alldonors_nostress_{celltype}_RNA_Multiome_RankGenesALL.txt",
        index=False, sep="\t", quoting=csv.QUOTE_ALL,
    )


          x
0      DTNA
1  SERPINE1
2      FAT1
3    KLHL29
4     ANXA2
          x
0      CPS1
1       HAL
2  PPARGC1A
3       SDS
4    INSIG1
      x
0   FGG
1   FGB
2    HP
3  SAA2
4    C3
         x
0  SLCO1B3
1   CYP3A4
2  SLCO1B7
3  ZNF385D
4   CYP2E1


## Build the top-50 marker gene list for Tangram

In [22]:
markers_df_adata_mash_heps_top50 = pd.DataFrame(adata_mash_heps.uns["rank_genes_groups"]["names"]).iloc[0:50, :]
markers_df_adata_mash_heps_top50.head()


,Hepatocytes.Senescent,Hepatocytes.Zone1,Hepatocytes.Zone2,Hepatocytes.Zone3
0,DTNA,CPS1,FGG,SLCO1B3
1,SERPINE1,HAL,FGB,CYP3A4
2,FAT1,PPARGC1A,HP,SLCO1B7
3,KLHL29,SDS,SAA2,ZNF385D
4,ANXA2,INSIG1,C3,CYP2E1


In [23]:
markers_df_adata_mash_heps_top50.shape # Check out dimensions

(50, 4)

In [25]:
# Create a flat array of the unique markers across all cell types.
markers_df_adata_mash_heps_top50 = list(np.unique(markers_df_adata_mash_heps_top50.melt().value.values))
len(markers_df_adata_mash_heps_top50)


195

## Export the reference object and marker list

In [26]:
# Needed to save the single-cell reference correctly.
adata_mash_heps.__dict__['_raw'].__dict__['_var'] = adata_mash_heps.__dict__['_raw'].__dict__['_var'].rename(columns={'_index': 'features'})


In [28]:
gene_nums = 50

export_dir = '/tscc/projects/ps-epigen/users/cmiciano/Liver/RNA/outputs/sandbox/spatial/250214_tangram_preprocessing_genelist50_liver_mash_heps_alldonors_top50_nostress/'
os.makedirs(export_dir, exist_ok=True)
print(export_dir)


/tscc/projects/ps-epigen/users/cmiciano/Liver/RNA/outputs/sandbox/spatial/250214_tangram_preprocessing_genelist50_liver_mash_heps_alldonors_top50_nostress/


In [30]:
# Write out the preprocessed reference + marker list, to be fed into the
# SLURM Tangram pipeline scripts.
adata_mash_heps.write(export_dir + 'Preprocessed_adsc.h5ad')

with open(export_dir + 'markers', 'w') as f:
    for item in markers_df_adata_mash_heps_top50:
        f.write(item + "\n")
